# 5. PyTorch 2导出量化

- pytorch 2导出量化主要过程如下：
    - 模型导出：使用torch.export.export()将模型转换为FX图表示
    - 量化器选择：
        - X86InductorQuantizer：适用于CPU部署
        - XNNPACKQuantizer：适用于移动端/ARM设备
    - 模型量化：
        - prepare：准备量化，在图上标注需要量化的节点
        - calibrate：校准，收集激活值的统计信息
        - convert：转换为实际的量化模型
    - 验证分析：
        - 精度对比
        - 模型大小对比
        - 性能对比
    - 自定义量化配置：
        - 可以自定义量化参数如数据类型、量化范围、量化方案等

## 5.1. PyTorch 2导出量化应用

### (1) 导入模块

In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.ao.quantization as tq
import torch.ao.quantization.quantize_pt2e as quantize_pt2e
from torch.ao.quantization.quantizer.x86_inductor_quantizer import X86InductorQuantizer
from torch.ao.quantization.quantizer.xnnpack_quantizer import XNNPACKQuantizer
import copy
import os
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageNet
from torch.utils.data import DataLoader

### (2) 加载数据集

In [2]:
# 返回DataLoader对象。
def load_data(root="F:/04Datasets/ImageNet2012", split="val"):
    # 加载数据集
    ds_imagenet2012 = ImageNet(
        root=root,
        split=split,
        transform = torchvision.models.ViT_H_14_Weights.IMAGENET1K_SWAG_E2E_V1.transforms() # 需要是对象
        # target_transform=None,   # 标签转换
        # loader=Image.open   # 默认（还可以直接加载为Tensor：）
    )
    # 取部分子集
    num_calibration = 1000   # 总样本是50000 
    num_calibration = num_calibration if num_calibration<=len(ds_imagenet2012) else len(ds_imagenet2012)
    torch.manual_seed(42)
    indices = torch.randperm(len(ds_imagenet2012))[:num_calibration] + 1  # +1是因为randperm生成0-999
    subsets_imagenet2012 = torch.utils.data.Subset(ds_imagenet2012, indices)

    loader_imagenet2012 = DataLoader(
        dataset=subsets_imagenet2012,        # 单样本数据集
        batch_size=100,   # 数据集批次大小
        shuffle=False,  # 是否随机洗牌数据集 
    )
    return loader_imagenet2012

In [3]:
loader_imagenet = load_data()
print(len(loader_imagenet))

10


### (3) 加载模型

In [4]:
def load_model():
    # model_ = torchvision.models.vit_h_14(torchvision.models.ViT_H_14_Weights.IMAGENET1K_SWAG_E2E_V1)
    model_ = torchvision.models.alexnet(torchvision.models.AlexNet_Weights.IMAGENET1K_V1)
    return model_


In [5]:
model_fp32 = load_model()
# model_fp32

### (4) 模型量化

In [6]:
# 4. 主要的量化流程
def quantize_model_pt2e(model, loader):
    original_model = copy.deepcopy(model)
    example_inputs = (list(loader)[0][0], )

    original_model.eval()
    original_model.to("cpu")
    
    # 1: 导出模型到FX图表示
    print("\n[步骤1] 导出模型到FX图...")
    # 使用torch.export导出模型
    exported_model = torch.export.export(   # export_for_training(  # export
        original_model, 
        example_inputs
    )
    print(f"导出成功! 图节点数: {len(exported_model.graph_module.graph.nodes)}")
    exported_model = exported_model.module()
    # exported_model.eval()
    # exported_model.to("cpu")
    # 2: 选择量化配置
    print("\n[步骤2] 配置量化器...")
    # 方法A: 使用X86InductorQuantizer (适用于CPU)
    x86_quantizer = X86InductorQuantizer()
    # 设置量化配置 - 静态量化
    x86_quantizer.set_global(tq.quantizer.x86_inductor_quantizer.get_default_x86_inductor_quantization_config())
    
    # 准备量化（在图上标注需要量化的节点）.
    print("\t|- 准备模型...")
    prepared_x86_model = quantize_pt2e.prepare_pt2e(
        exported_model,
        x86_quantizer
    )
    
    # 校准（使用一些样本数据进行校准）
    print("\t|- 正在进行校准...")
    for i in range(1):
        for x, _ in loader:
            prepared_x86_model(x)
            break
        break
    
    # 转换到量化模型
    print("\t|- 开始量化")
    quantized_x86_model = quantize_pt2e.convert_pt2e(prepared_x86_model)
    print("\t|- X86量化完成!")
    return quantized_x86_model

In [7]:
quantized_model =  quantize_model_pt2e(model_fp32, loader_imagenet)
quantized_model


[步骤1] 导出模型到FX图...
导出成功! 图节点数: 40

[步骤2] 配置量化器...
	|- 准备模型...
	|- 正在进行校准...
	|- 开始量化
	|- X86量化完成!


GraphModule(
  (features): Module(
    (0): Module()
    (3): Module()
    (6): Module()
    (8): Module()
    (10): Module()
  )
  (classifier): Module(
    (1): Module()
    (4): Module()
    (6): Module()
  )
)

### (5) 验证量化模型的准确率

In [10]:
def validate_model(model, test_loader, device="cpu"):
    model.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            y_ = model(x)
            _, pred = y_.max(1)
            total += y.size(0)
            correct += (pred==y).sum().item()
    accuracy = 100. * correct / total
    return accuracy

In [11]:
validate_model(quantized_model, loader_imagenet)

51.4

## 5.2. PyTorch2导出量化的性能分析